### 1.5 List Available Runtimes
Call `ae.find_runtimes()` to list all HPC resources you have access to.

# Cybershuttle SDK -  Molecular Dynamics
> Plan, distribute, monitor, and analyze NAMD experiments across HPC runtimes.

This notebook demonstrates how to plan, launch, monitor, and analyze **NAMD** experiments with replicas through the Cybershuttle SDK.

## 1. Preliminaries

### 1.1 Install the SDK

Install the Airavata Python SDK to orchestrate NAMD experiments directly from JupyterLab. Requires Python 3.10+.

In [ ]:
%pip install -qU "airavata-python-sdk[notebook]==2.2.0"

### 1.2 Point the SDK to Cybershuttle

Configure which Cybershuttle deployment you’re targeting. Defaults to Production.

In [ ]:
import os

# Production
os.environ['AUTH_SERVER_URL'] = "https://auth.cybershuttle.org"
os.environ['API_SERVER_HOSTNAME'] = "api.gateway.cybershuttle.org"
os.environ['GATEWAY_URL'] = "https://gateway.cybershuttle.org"
os.environ['STORAGE_RESOURCE_HOST'] = "gateway.cybershuttle.org"

# # Development
# os.environ['AUTH_SERVER_URL'] = "https://auth.dev.cybershuttle.org"
# os.environ['API_SERVER_HOSTNAME'] = "api.dev.cybershuttle.org"
# os.environ['GATEWAY_URL'] = "https://gateway.dev.cybershuttle.org"
# os.environ['STORAGE_RESOURCE_HOST'] = "gateway.dev.cybershuttle.org"

### 1.3 Import Packages

Load the core API and MD app bindings (NAMD, AMBER, GROMACS, etc.).

In [ ]:
import airavata_experiments as ae
import airavata_experiments.md

### 1.4 Authenticate

Call `ae.login()` to generate a one-time login link to sign in with your institution or email.

In [ ]:
ae.login()

In [ ]:
runtimes = ae.find_runtimes(group="MDWorkshop")
ae.display(runtimes)

## 2. Run a NAMD Experiment

We preloaded a sample pull‐simulation under `data/namd`.
You can also bring in (drag-drop) your own files to run experiments.

```bash
data/namd
├── b4pull.pdb
├── b4pull.restart.coor
├── b4pull.restart.vel
├── b4pull.restart.xsc
├── par_all36_water.prm
├── par_all36m_prot.prm
├── pull_cpu.conf
├── structure.pdb
└── structure.psf

```

### 2.1 Create Experiment

Define your NAMD experiment by calling `ae.md.NAMD.initialize()` and providing the paths to your `.conf`, `.pdb`, `.psf`, and other files.
> If your IDE supports auto-completion, it will show you the method signature.

```python
def initialize(
    name: str,
    config_file: str,
    pdb_file: str,
    psf_file: str,
    ffp_files: list[str],
    other_files: list[str] = [],
    parallelism: Literal['CPU', 'GPU'] = "CPU",
    num_replicas: int = 1
) -> Experiment[ExperimentApp]
```

To add replica runs, call `exp.add_run()` once per replica -- you can optionally specify runtime and resource constraints for each run.

To perform parameter sweeps, iterate over your parameter space and call `exp.add_run()` with each parameter set as keyword arguments.

In [ ]:
exp = ae.md.NAMD.initialize(
    name="SMD",
    config_file="data/namd/pull_gpu.conf",
    pdb_file="data/namd/structure.pdb",
    psf_file="data/namd/structure.psf",
    ffp_files=[
      "data/namd/par_all36_water.prm",
      "data/namd/par_all36m_prot.prm"
    ],
    other_files=[
      "data/namd/b4pull.pdb",
      "data/namd/b4pull.restart.coor",
      "data/namd/b4pull.restart.vel",
      "data/namd/b4pull.restart.xsc",
    ],
    parallelism="GPU",
)
runtimes = ae.find_runtimes(group="MDWorkshop", cluster="NCSADelta")
for _ in range(1):
    exp.add_run(use=runtimes, cpus=8, nodes=1, walltime=60)

Call `ae.display(exp)` to print the experiment details.

In [ ]:
ae.display(exp)

### 2.2 Build Execution Plan

Call `exp.plan()` to upload your inputs to Cybershuttle and create a reproducible plan that's runnable from anywhere.

Call `ae.display(plan)` to print the plan details.

In [ ]:
plan = exp.plan()
ae.display(plan)

Call `plan.export()` to export the plan (as JSON) for future reference.

In [ ]:
plan.export("plan_gpu.json")

### 2.3 Launch Experiment

Call `plan.launch()` to launch the experiment.
The SDK will record tracking metadata (job IDs, working directories, etc.) and update the plan state in Cybershuttle.
> If you exported your plan before, rerun `plan.export()` to refresh your local JSON.

In [ ]:
plan.launch()
plan.export("plan_gpu.json")

### 2.4 Monitor Experiment

Call `plan.status()` to check the current state of the experiment and its runs. You can poll for experiment completion by calling this periodically.

In [ ]:
plan.status()

Call `plan.stop()` to stop the entire experiment (all runs). To stop specific runs, pass their indices as a `runs` argument.

e.g., `plan.stop(runs=[a,b])` stops only the $a^{th}$ and $b^{th}$ runs; others, if any, will continue to run.

In [ ]:
plan.stop()
plan.stop(runs=[0])

You can loop over `plan.tasks` to interact with each run in real time.
The SDK provides helper functions to print metadata, list files, transfer files, preview file content, and run shell commands on the fly.

Each `task` object has several helper functions to perform file operations within its context.

* `task.ls()` - list all remote files (inputs, outputs, logs, etc.)
* `task.upload(<local_path>, <remote_path>)` - upload a local file to remote
* `task.cat(<remote_path>)` - displays contents of a remote file
* `task.download(<remote_path>, <local_path>)` - fetch a remote file to local

In [ ]:
for task in plan.tasks:
    print(task.name, task.pid, task.workdir)
    display(task.ls())                                      # list files
    task.upload("data/sample.txt")                          # upload sample.txt
    display(task.cat("sample.txt"))                         # preview sample.txt
    task.exec("cat sample.txt | xargs wc -l > count.log")   # generate count.log
    task.download("count.log", f"./results_{task.name}")    # download count.log

Call `plan.wait_for_completion()` to block further execution until the experiment completes.

In [ ]:
plan.wait_for_completion()

## 3. Analyze Experiment Runs


### 3.1 List All Experiments

Call `ae.plan.query()` to retrieve all plans you created through the SDK.

In [ ]:
plans = ae.plan.query()
ae.display(plans)

You can find a plan, record its id, and call `ae.plan.load(plan_id)` with that id to load it into memory.
Alternatively, if you exported a plan before, call `ae.plan.load_json(path_to_plan)` to load that instead.

In [ ]:
plan = ae.plan.load_json("plan_gpu.json")
plan = ae.plan.load(plan.id)
ae.display(plan)

### 3.2 Process Experiment Results Interactively

You can launch an interactive job where the plan was executed, without requiring extra configuration.
The `--state=<path/to/plan/file>` argument will bring those results into the interactive job.
All results will be stored in a `<project_name>_results`

In [1]:
import airavata_jupyter_magic

%request_runtime hpc --file=cybershuttle.yml --walltime=60 --plan=plan_gpu.json
%wait_for_runtime hpc --live
%switch_runtime hpc


Loaded airavata_jupyter_magic (2.1.7) 
(current runtime = local)

  %authenticate                              -- Authenticate to access high-performance runtimes.
  %request_runtime <rt> [args]               -- Request a runtime named <rt> with configuration <args>.
                                                Call multiple times to request multiple runtimes.
  %restart_runtime <rt>                      -- Restart runtime <rt> if it hangs. This will clear all variables.
  %stop_runtime <rt>                         -- Stop runtime <rt> when no longer needed.
  %wait_for_runtime <rt>                     -- Wait for runtime <rt> to be ready.
  %switch_runtime <rt>                       -- Switch the active runtime to <rt>. All subsequent cells will run here.
  %%run_on <rt>                              -- Force a cell to always execute on <rt>, regardless of the active runtime.
  %stat_runtime <rt>                         -- Show the status of runtime <rt>.
  %copy_data source=<r1:f1

Output()

Authenticated.

Requesting runtime=hpc...
[NCSADelta:gpuA100x4, 60 Minutes, 1 Node(s), 4 CPU(s), 0 GPU(s), 4096 MB RAM, 0 MB VRAM]
* modules=[]
* libraries=['python=3.10', 'pip']
* pip=['numpy', 'pandas']
* mounts=[]
* links={'PROCESS_c94be6ce-34c7-4623-b557-ef0d7551d9c9': 'SMD_4141'}
Requested runtime=hpc
Request successful: runtime=hpc


Output()

InvalidStateError: Runtime=hpc is in state=EXPERIMENT_COMPLETED
process  completed

STDOUT:
CS_HOME=/u/svcscigapgwuser/cybershuttle
AGENT=agent_56e7f813-cda6-4d5f-b3a8-839844e1fb35
SERVER=api.gateway.cybershuttle.org
CONTAINER=
LIBRARIES=python=3.10,pip
PIP=numpy,pandas
MOUNTS=
receiving incremental file list
Plasma-Vlab-amitava-class/
Plasma-Vlab-gemReconnection/

sent 4,818 bytes  received 3,703,551,142 bytes  42,815,675.84 bytes/sec
total size is 43,799,049,932  speedup is 11.83

STDERR:
% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0100   120    0     0  100   120      0   1224 --:--:-- --:--:-- --:--:--  1224
rsync: recv_generator: mkdir "/u/svcscigapgwuser/cybershuttle/dataset/Plasma-Vlab-amitava-class" failed: Disk quota exceeded (122)
*** Skipping any contents from this failed directory ***
r

In [ ]:
%%bash

echo $pwd
ls -lrt .

In [ ]:
%stop_runtime hpc